## 安裝套件

In [1]:
!pip install -q \
transformers \
accelerate \
bitsandbytes \
pillow \
langchain \
langchain-community \
langchain-text-splitters \
chromadb \
sentence-transformers \
huggingface_hub

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 14.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 33.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 56.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 26.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 83.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 44.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 552.2/552.2 kB 36.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 71.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.8/71.8 kB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.9/170.9 kB 16.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.7/203

In [2]:
import os
import torch
from PIL import Image

# 建立 RAG 資料庫

In [13]:
from langchain_core.documents import Document
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings

# 讀取食譜
with open("recipes.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()

# 一道料理切成一個 Document
recipes = raw_text.split("---------------------------------------------------")

documents = [
    Document(page_content=r.strip())
    for r in recipes
    if r.strip()
]

embedding = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

db = Chroma.from_documents(
    documents,
    embedding,
    persist_directory="./recipe_db"
)

print("RAG資料庫建立完成")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

RAG資料庫建立完成


# 輔助函數

## 食材標準化

In [14]:
ingredient_map = {
    "小農鮮乳": "牛奶",
    "鮮乳": "牛奶",
    "全脂鮮乳": "牛奶",
    "全脂牛奶": "牛奶"
}

In [15]:
def normalize_ingredients(ingredients):

    result = set()

    for item in ingredients:

        item = item.strip()

        if item in ingredient_map:
            item = ingredient_map[item]

        result.add(item)

    return result

## 食材解析

In [16]:
def parse_ingredients(text):

    lines = text.split("\n")

    ingredients = []

    capture = False

    for line in lines:

        line = line.strip()

        if "需要食材" in line:
            capture = True
            continue

        if "做法" in line:
            break

        if capture and line:
            ingredients.append(line)

    return set(ingredients)

## 取得料理名稱

In [17]:
def get_recipe_name(text):

    for line in text.split("\n"):

        line = line.strip()

        if line.startswith("料理："):
            return line.replace("料理：", "").strip()

    return "未知料理"

## 缺少食材評分 (重新排序)

In [18]:
def rerank(recipes, user_ingredients):

    scored = []

    for doc in recipes:

        recipe_ing = parse_ingredients(
            doc.page_content
        )

        matched = len(
            recipe_ing & user_ingredients
        )

        missing = (
            recipe_ing - user_ingredients
        )

        score = len(missing)

        scored.append({
            "score": score,
            "matched": matched,
            "missing": missing,
            "text": doc.page_content
        })

    scored.sort(
        key=lambda x: (
            x["score"],
            -x["matched"]
        )
    )

    return scored

## Pseudo Query

In [19]:
def rewrite_query(ingredients):
    return f"根據冰箱現有食材 {ingredients} 推薦可製作的料理"

# 載入 VLM 模型

In [21]:
import os
from google.colab import userdata
from huggingface_hub import login

HF_TOKEN = userdata.get(
    "HUGGINGFACE_ACCESS_TOKEN"
)

login(HF_TOKEN)

In [22]:
from transformers import Qwen2VLForConditionalGeneration, AutoProcessor

model_id = "Qwen/Qwen2-VL-2B-Instruct"

processor = AutoProcessor.from_pretrained(model_id)

model = Qwen2VLForConditionalGeneration.from_pretrained(
    model_id,
    torch_dtype=torch.float16,
    device_map="auto"
)

print("VLM載入完成")

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/729 [00:00<?, ?it/s]

VLM載入完成


## 提取食材

In [23]:
def extract_ingredients(image_path):
    image = Image.open(image_path)

    messages = [{
        "role": "user",
        "content": [
            {"type": "image", "image": image},
            {"type": "text",
             "text": "請辨識圖片中的食材，只輸出食材名稱，用逗號分隔。"}
        ]
    }]

    text = processor.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = processor(text=[text], images=[image], return_tensors="pt").to(model.device)

    outputs = model.generate(**inputs, max_new_tokens=50)

    result = processor.decode(outputs[0], skip_special_tokens=True)

    if "assistant" in result:
        result = result.split("assistant")[-1]

    return result.strip()

# Full Pipeline

In [31]:
image_path = "fridge.png"

# ==================================================
# Step 1 VLM 食材辨識
# ==================================================

ingredients_raw = extract_ingredients(image_path)

print("辨識結果：", ingredients_raw)

user_ingredients = normalize_ingredients(
    [
        i.strip()
        for i in ingredients_raw.split(",")
    ]
)

print("標準化後：", user_ingredients)

# ==================================================
# Step 2 Query Rewrite
# ==================================================

query = rewrite_query(
    ingredients_raw
)


# ==================================================
# Step 4 Retrieval
# ==================================================

docs = db.similarity_search(
    query,
    k=3
)

# ==================================================
# Step 5 Rerank
# ==================================================

ranked = rerank(
    docs,
    user_ingredients
)

best_recipe = ranked[0]

recipe_name = get_recipe_name(
    best_recipe["text"]
)

# ==================================================
# Debug
# ==================================================

print("\n=== Ranking ===")

for r in ranked:

    print(
        get_recipe_name(r["text"]),
        "缺少:",
        len(r["missing"]),
        r["missing"]
    )

# ==================================================
# Step 6 建議生成
# ==================================================

# 取得缺少食材
missing_list = list(best_recipe["missing"])

recipe_name = get_recipe_name(
    best_recipe["text"]
)

if len(missing_list) == 0:

    print("\n🍽️ 最終推薦")
    print(f"最適合料理：{recipe_name}")
    print("缺少食材：無")
    print("解決方法建議：食材已齊全，可以直接製作。")
else:

    missing_text = "、".join(missing_list)

    messages = [
        {
            "role": "system",
            "content":
            "你是一位料理助手。"
        },
        {
            "role": "user",
            "content":
            f"""
料理名稱：{recipe_name}

缺少食材：
{missing_text}

請提供一句簡短建議，以解決缺少食材的問題。

只輸出建議內容。
不要重複料理名稱。
不要解釋。
"""
        }
    ]

    text = processor.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = processor(
        text=[text],
        return_tensors="pt"
    ).to(model.device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=40,
        do_sample=False
    )

    generated_ids = outputs[
        :,
        inputs.input_ids.shape[1]:
    ]

    suggestion = processor.batch_decode(
        generated_ids,
        skip_special_tokens=True
    )[0].strip()

    print("\n🍽️ 最終推薦")
    print(f"最適合料理：{recipe_name}")
    print(f"缺少食材：{missing_text}")
    print(f"解決方法建議：{suggestion}")

辨識結果： 小農鮮乳, 番茄
標準化後： {'番茄', '牛奶'}

=== Ranking ===
番茄炒蛋 缺少: 1 {'雞蛋'}
奶油濃湯 缺少: 2 {'奶油', '洋蔥'}
布丁 缺少: 2 {'雞蛋', '糖'}

🍽️ 最終推薦
最適合料理：番茄炒蛋
缺少食材：雞蛋
解決方法建議：建議您使用其他食材代替雞蛋，如豆腐、火腿等。
